# SE4_generate_outputs


<span style='color:red'>delete before publishing</span>

# About this notebook (for developer, delete this cell)

This is the final step in the DIWA reproducible research pipeline. SE4 reads predictions and metrics written by SE3 and generates **publication-quality figures and summary tables** for your manuscript, thesis, or report. No model training or data processing occurs here.

**Key design rules (do not violate):**
- All inputs are read from `autobag_processeddata/predictions/`.
- All outputs (figures, tables) are written to `autobag_processeddata/outputs/`.
- No data is held in memory between SE notebooks.
- Figures follow the ten visualisation guidelines of **Kelleher & Wagener (2011)**.

**Kelleher & Wagener (2011) — Ten guidelines for effective data visualisation:**

| # | Guideline | Practical implication |
|---|---|---|
| 1 | Simplest graph that conveys the message | Remove all non-data ink; no 3D; no decorative elements |
| 2 | Choose encoding objects and attributes carefully | Use position/length for quantitative data; colour only for categories or patterns |
| 3 | Focus on patterns OR details, not both | Heatmaps/FDCs for patterns; line/bar for details |
| 4 | Use appropriate scale | Log scale for streamflow; avoid truncated y-axes |
| 5 | Use colour appropriately | Colourblind-safe palettes; sequential for magnitude, diverging for anomalies |
| 6 | Label all axes with units | Include variable name and unit in every axis label |
| 7 | Provide context | Add reference lines (baseline, target thresholds, climatology) |
| 8 | Maintain consistency | Same palette, font size, and axis style across all figures in a paper |
| 9 | Use small multiples for comparisons | Panel plots instead of overloaded single-axis figures |
| 10 | Tailor figure to audience | Journal figures differ from slide figures; adapt accordingly |

**Recommended packages:**

| Purpose | Packages |
|---|---|
| Static publication figures | `matplotlib`, `seaborn` |
| Interactive / exploratory | `plotly`, `hvplot`, `bokeh` |
| Geospatial maps | `cartopy`, `geopandas`, `contextily` |
| Colour palettes | `cmocean`, `palettable`, `matplotlib` |
| Tables | `pandas`, `tabulate` |
| Report generation | `nbconvert`, `jinja2` |

In [ ]:
#Modify cell
# Load dependencies — keep only what you use, add what you need
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

# Optional — install as needed
# import cmocean                      # pip install cmocean  (perceptually uniform colormaps)
# import seaborn as sns               # pip install seaborn
# import cartopy.crs as ccrs          # pip install cartopy  (geospatial maps)
# import contextily as ctx            # pip install contextily (basemap tiles)

# Consistent figure style across all SE4 outputs
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

<span style='color:goldenrod'>edit before publishing</span>

# Overview

## Purpose of this notebook
Describe what publication outputs this notebook generates.

## Inputs
- `predictions.csv` — observed and predicted values from SE3
- `evaluation_metrics.csv` — performance metrics from SE3
- `model_config.yaml` — model configuration from SE3
- *(add any other files read from `processeddata/`)*

## Outputs
All figures and tables are written to `autobag_processeddata/outputs/`.

| Output file | Figure / Table | Description |
|---|---|---|
| `fig01_hydrograph.png` | Figure 1 | Observed vs. predicted hydrograph |
| `fig02_scatter.png` | Figure 2 | Scatter plot across all splits |
| `fig03_fdc.png` | Figure 3 | Flow Duration Curve |
| `fig04_heatmap_monthly.png` | Figure 4 | Monthly performance heatmap |
| `table01_metrics.csv` | Table 1 | Model performance metrics |
| *(add rows as needed)* | | |

<span style='color:green'>keep</span>

## Directory setup
All inputs are read from the processed data directory; all outputs are written to the outputs subdirectory outside the repository.

In [ ]:
# Automatically generated cell — resolve external directories
# Current working directory = project/notebooks
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
external_base = project_root.parent

# Read from processed data directory (written by SE2 and SE3)
proc_data_dir = external_base / "autobag_processeddata"
pred_dir      = proc_data_dir / "predictions"
model_dir     = proc_data_dir / "model"
config_dir    = model_dir / "config"

# Write all publication outputs here
output_dir    = proc_data_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print("Reading from:", proc_data_dir.resolve())
print("Writing to  :", output_dir.resolve())

In [ ]:
!df -k ..
# Confirm sufficient storage is available before writing outputs

<span style='color:red'>delete before publishing</span>

# Step 1 — Load all inputs from disk

All data are loaded from files. Never re-run model training here; SE4 is read-only with respect to SE3 outputs.

In [ ]:
#Modify cell
# Load model config
with open(config_dir / "model_config.yaml") as f:
    cfg = yaml.safe_load(f)

target = cfg["target_variable"]

# Load predictions
df_pred = pd.read_csv(pred_dir / "predictions.csv", parse_dates=["date"], index_col="date")

# Load evaluation metrics
metrics_df = pd.read_csv(pred_dir / "evaluation_metrics.csv", index_col="split")

# Convenience split subsets
train = df_pred[df_pred["split"] == "train"]
val   = df_pred[df_pred["split"] == "validation"]
test  = df_pred[df_pred["split"] == "test"]

print(f"Predictions loaded  : {len(df_pred)} rows")
print(f"Metrics loaded      : {metrics_df.shape}")
print("\nPerformance summary:")
print(metrics_df.to_string())

<span style='color:red'>delete before publishing</span>

# Step 2 — Publication-quality hydrograph (Figure 1)

Guideline 1: simplest graph. Guideline 3: focus on temporal pattern. Guideline 6: axis labels with units. Guideline 8: consistent style. Guideline 9: small multiples for train/val/test context.

In [ ]:
#Modify cell
# ── Figure 1: Full-period hydrograph with split shading ───────────────────────
fig, ax = plt.subplots(figsize=(11, 3.5))

# Shade split periods (Guideline 7: provide context)
ax.axvspan(train.index.min(), train.index.max(), alpha=0.07, color="#4dac26", label="Train period")
ax.axvspan(val.index.min(),   val.index.max(),   alpha=0.10, color="#f1a340", label="Val period")
ax.axvspan(test.index.min(),  test.index.max(),  alpha=0.10, color="#d01c8b", label="Test period")

# Observed and predicted lines (Guideline 2: line for continuous temporal data)
ax.plot(df_pred.index, df_pred["observed"],  color="#2166ac", lw=1.2, label="Observed",  zorder=4)
ax.plot(df_pred.index, df_pred["predicted"], color="#d6604d", lw=0.9, label="Predicted", zorder=3, alpha=0.9)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.set_xlabel("Date")
ax.set_ylabel(f"{target}")     # Guideline 6: units must be specified by student
ax.set_title(f"Fig. 1 — Simulated vs. observed {target}  |  test NSE={metrics_df.loc['test','NSE']:.3f}  KGE={metrics_df.loc['test','KGE']:.3f}")
ax.legend(ncol=5, frameon=False, loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig(output_dir / "fig01_hydrograph.png")
plt.show()
print("Figure 1 saved")

<span style='color:red'>delete before publishing</span>

# Step 3 — Observed vs. predicted scatter (Figure 2)

Guideline 2: scatter uses position to encode quantitative accuracy. Guideline 5: colourblind-safe split colours. Guideline 9: single panel — do not stack multiple scatter in same axis without clear separation.

In [ ]:
#Modify cell
# ── Figure 2: Observed vs. predicted scatter ──────────────────────────────────
split_colours = {"train": "#4dac26", "validation": "#f1a340", "test": "#d01c8b"}

fig, ax = plt.subplots(figsize=(4.5, 4.5))
for sp, col in split_colours.items():
    mask = df_pred["split"] == sp
    nse  = metrics_df.loc[sp, "NSE"] if sp in metrics_df.index else float("nan")
    ax.scatter(
        df_pred.loc[mask, "observed"], df_pred.loc[mask, "predicted"],
        s=5, alpha=0.4, color=col, label=f"{sp} (NSE={nse:.3f})", rasterized=True,
    )

lim_min = df_pred[["observed", "predicted"]].min().min()
lim_max = df_pred[["observed", "predicted"]].max().max()
ax.plot([lim_min, lim_max], [lim_min, lim_max], "k--", lw=0.8, label="1:1")   # Guideline 7
ax.set_xlabel(f"Observed {target}")
ax.set_ylabel(f"Predicted {target}")
ax.set_title("Fig. 2 — Observed vs. Predicted")
ax.legend(frameon=False, markerscale=2.5, fontsize=8)
plt.tight_layout()
plt.savefig(output_dir / "fig02_scatter.png")
plt.show()
print("Figure 2 saved")

<span style='color:red'>delete before publishing</span>

# Step 4 — Flow Duration Curve (Figure 3)

Guideline 3: FDC is a pattern plot — reveals how well the model captures the statistical distribution of flows across the full range. Guideline 4: log-y scale is standard for streamflow spanning multiple orders of magnitude.

In [ ]:
#Modify cell
# ── Figure 3: Flow Duration Curve — test period ───────────────────────────────
def fdc(series):
    s = np.sort(np.asarray(series))[::-1]
    p = np.arange(1, len(s) + 1) / (len(s) + 1)
    return p * 100, s

fig, ax = plt.subplots(figsize=(5.5, 4))
exc_o, q_o = fdc(test["observed"])
exc_p, q_p = fdc(test["predicted"])
ax.semilogy(exc_o, q_o, color="#2166ac", lw=1.4, label="Observed")
ax.semilogy(exc_p, q_p, color="#d6604d", lw=1.1, ls="--", label="Predicted")
ax.set_xlabel("Exceedance probability (%)")
ax.set_ylabel(f"{target} (log scale)")    # Guideline 6
ax.set_title("Fig. 3 — Flow Duration Curve — test period")
ax.legend(frameon=False)
# Guideline 7: annotate high / medium / low flow regimes
ax.axvline(10,  color="gray", lw=0.6, ls=":");  ax.text(10.5,  ax.get_ylim()[1]*0.7, "High",  fontsize=7, color="gray")
ax.axvline(90,  color="gray", lw=0.6, ls=":");  ax.text(78,    ax.get_ylim()[1]*0.7, "Low",   fontsize=7, color="gray")
plt.tight_layout()
plt.savefig(output_dir / "fig03_fdc.png")
plt.show()
print("Figure 3 saved")

<span style='color:red'>delete before publishing</span>

# Step 5 — Monthly residual heatmap (Figure 4)

Guideline 3: heatmap is ideal for revealing seasonal patterns in bias. Guideline 5: use a **diverging** colour scale centred on zero — blue = underestimation, red = overestimation. Do not use rainbow (jet) colormaps.

In [ ]:
#Modify cell
# ── Figure 4: Monthly bias heatmap — test period ──────────────────────────────
test_copy = test.copy()
test_copy["residual"] = test_copy["predicted"] - test_copy["observed"]
test_copy["year"]     = test_copy.index.year
test_copy["month"]    = test_copy.index.month

pivot = test_copy.pivot_table(values="residual", index="year", columns="month", aggfunc="mean")
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

vmax = np.nanpercentile(np.abs(pivot.values), 95)   # symmetric colour scale
fig, ax = plt.subplots(figsize=(9, max(3, len(pivot) * 0.4)))
im = ax.imshow(pivot.values, cmap="RdBu_r", aspect="auto",   # Guideline 5: diverging
               vmin=-vmax, vmax=vmax, interpolation="nearest")
cbar = fig.colorbar(im, ax=ax, pad=0.02, shrink=0.8)
cbar.set_label(f"Mean bias (predicted − observed) {target}")  # Guideline 6
ax.set_xticks(range(12))
ax.set_xticklabels(month_labels)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index)
ax.set_title("Fig. 4 — Monthly mean bias (test period)")       # Guideline 3: pattern
plt.tight_layout()
plt.savefig(output_dir / "fig04_heatmap_monthly.png")
plt.show()
print("Figure 4 saved")

<span style='color:red'>delete before publishing</span>

# Step 6 — Performance metrics panel (Figure 5)

Guideline 9: small multiples — one panel per metric, consistent scale. Guideline 2: horizontal bars encode magnitude; position makes comparison easy. Guideline 7: add reference lines at NSE/KGE = 0 (no-skill) and 1 (perfect).

In [ ]:
#Modify cell
# ── Figure 5: Metrics summary bar panel ───────────────────────────────────────
plot_metrics = ["NSE", "KGE", "logNSE", "PBIAS_%", "RMSE", "R2"]
plot_splits  = [s for s in ["train", "validation", "test"] if s in metrics_df.index]
ref_lines    = {"NSE": 0, "KGE": 0, "logNSE": 0, "R2": 0}   # no-skill reference

n_metrics = len(plot_metrics)
split_colours = {"train": "#4dac26", "validation": "#f1a340", "test": "#d01c8b"}

fig, axes = plt.subplots(1, n_metrics, figsize=(n_metrics * 2, 3.5), sharey=True)
x = np.arange(len(plot_splits))

for ax, metric in zip(axes, plot_metrics):
    vals   = [metrics_df.loc[sp, metric] for sp in plot_splits]
    colours = [split_colours[sp] for sp in plot_splits]
    bars = ax.bar(x, vals, color=colours, edgecolor="none", width=0.6)
    if metric in ref_lines:
        ax.axhline(ref_lines[metric], color="k", lw=0.7, ls="--")   # Guideline 7
    ax.set_title(metric, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([s[:3].capitalize() for s in plot_splits], fontsize=8)
    ax.tick_params(axis="y", labelsize=8)

fig.suptitle("Fig. 5 — Model performance metrics by split", fontsize=10, y=1.02)
# Add legend for splits
from matplotlib.patches import Patch
handles = [Patch(color=split_colours[s], label=s) for s in plot_splits]
fig.legend(handles=handles, frameon=False, loc="upper center",
           bbox_to_anchor=(0.5, -0.05), ncol=len(plot_splits), fontsize=8)
plt.tight_layout()
plt.savefig(output_dir / "fig05_metrics_panel.png")
plt.show()
print("Figure 5 saved")

<span style='color:red'>delete before publishing</span>

# Step 7 — Export Table 1: model performance metrics

Export a clean CSV and a formatted Markdown/LaTeX table suitable for direct insertion into a manuscript. The table includes all splits and the naïve baseline.

In [ ]:
#Modify cell
# ── Table 1: performance metrics ─────────────────────────────────────────────
table = metrics_df.copy()

# Save as CSV for archiving
table_csv = output_dir / "table01_metrics.csv"
table.to_csv(table_csv)
print(f"Table 1 CSV saved to {table_csv}")

# Print Markdown for README or supplement
print("\n--- Markdown (copy into README or supplement) ---")
print(table.to_markdown(floatfmt=".4f"))

# Optional: save LaTeX for manuscript
table_tex = output_dir / "table01_metrics.tex"
table.style.format(precision=4).to_latex(table_tex, caption="Model performance metrics.",
                                          label="tab:metrics", position="h")
print(f"\nLaTeX table saved to {table_tex}")

<span style='color:red'>delete before publishing</span>

# Step 8 — Additional figures (add as needed)

Add domain-specific figures below. Examples for water resources:

- **Spatial map** of model performance across catchments (use `cartopy` + `geopandas`)
- **Uncertainty bands** around predictions (plot 5th/95th percentile if ensemble or Bayesian model)
- **Trend plot** for hypothesis-testing results (Mann-Kendall slope + confidence interval)
- **Exceedance frequency** of extreme events (return period plot)
- **Sensitivity / SHAP values** for feature attribution
- **Seasonal decomposition** of residuals

Refer to Kelleher & Wagener (2011) before choosing a plot type:
- Pattern question → heatmap, FDC, horizon graph
- Detail question → line graph, bar chart
- Distribution question → box plot, violin, empirical CDF
- Spatial question → choropleth, dot map (avoid 3D surface maps)

In [ ]:
#Modify cell — add your domain-specific figures here

# Example: seasonal box plot of residuals (pattern across months)
# Guideline 3: box plot reveals distributional patterns by month
# Guideline 9: 12-panel small-multiple or single grouped box

test_copy = test.copy()
test_copy["residual"] = test_copy["predicted"] - test_copy["observed"]
test_copy["month"]    = test_copy.index.month_name().str[:3]

month_order = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(10, 3.5))
groups = [test_copy.loc[test_copy["month"] == m, "residual"].dropna().values
          for m in month_order]
bp = ax.boxplot(groups, labels=month_order, patch_artist=True,
                medianprops=dict(color="black", lw=1.5),
                boxprops=dict(facecolor="#a6cee3", alpha=0.7),
                flierprops=dict(marker=".", ms=3, alpha=0.3))
ax.axhline(0, color="k", lw=0.8, ls="--")   # Guideline 7: zero-bias reference
ax.set_xlabel("Month")
ax.set_ylabel(f"Residual (predicted − observed) {target}")
ax.set_title("Fig. 6 — Seasonal distribution of model residuals — test period")
plt.tight_layout()
plt.savefig(output_dir / "fig06_seasonal_residuals.png")
plt.show()
print("Figure 6 saved")

<span style='color:goldenrod'>edit before publishing</span>

# Summary of outputs

Summarise the key findings communicated by each figure:
- What does Figure 1 show about model performance over time?
- Does Figure 2 reveal systematic over- or under-prediction?
- Does Figure 3 show the model captures both high and low flow regimes?
- Does Figure 4 reveal seasonal patterns in bias?
- How does model skill (Figure 5) compare to the naïve baseline?

## Files written by this notebook

| File | Description |
|---|---|
| `fig01_hydrograph.png` | Full-period hydrograph with split shading |
| `fig02_scatter.png` | Observed vs. predicted scatter by split |
| `fig03_fdc.png` | Flow duration curve — test period |
| `fig04_heatmap_monthly.png` | Monthly bias heatmap — test period |
| `fig05_metrics_panel.png` | Metrics bar panel across all splits |
| `fig06_seasonal_residuals.png` | Seasonal residual box plots |
| `table01_metrics.csv` | Performance metrics table |
| `table01_metrics.tex` | LaTeX-formatted metrics table |

## Need help?

Search, ask, and answer visualisation and output questions at https://github.com/orgs/DigitalWaters-fi/discussions

Tag #outputs